# 02 — DECA 3D Face Reconstruction

Notebook này minh họa quy trình tái tạo khuôn mặt 3D bằng **DECA** (Detailed Expression Capture and Animation).

## DECA là gì?

DECA là mô hình học sâu tái tạo 3D khuôn mặt từ **một ảnh đơn lẻ**, bao gồm:
- **Shape** — hình dạng tổng thể (100 tham số FLAME)
- **Pose** — góc đầu (6 DOF)
- **Expression** — biểu cảm khuôn mặt (50 tham số)
- **Texture** — màu sắc da (UV texture map)
- **Illumination** — điều kiện ánh sáng
- **Detail** — chi tiết da (nếp nhăn, lỗ chân lông)

## Yêu cầu
1. DECA submodule đã được khởi tạo: `git submodule update --init --recursive`
2. DECA dependencies đã cài: `pip install -r external/DECA/requirements.txt`
3. Pretrained weights đã tải: `bash scripts/download_weights.sh`

## Nội dung
1. Kiểm tra DECA installation
2. Chạy DECA inference
3. Xem kết quả: mesh, texture, rendered image
4. Load và inspect output parameters

## 1. Import và kiểm tra

In [ ]:
import sys
from pathlib import Path

ROOT = Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from src.recon.run_deca import validate_deca_installation, check_deca_weights, run_deca
from src.render.mesh_export import load_obj
from src.utils.io import ensure_dir, list_image_files
from src.utils.logger import get_logger

logger = get_logger('notebook_02')
print('✅ Import thành công!')

## 2. Kiểm tra DECA Installation

In [ ]:
DECA_ROOT = Path('external/DECA')
INPUT_IMAGE = Path('data/samples/test.jpg')
OUTPUT_DIR = Path('outputs')

ensure_dir(OUTPUT_DIR)

print(f'DECA_ROOT: {DECA_ROOT}')
print(f'DECA exists: {DECA_ROOT.exists()}')

demo_script = validate_deca_installation(DECA_ROOT)
if demo_script:
    print(f'✅ Demo script: {demo_script}')
else:
    print('❌ DECA chưa được cài đặt. Xem hướng dẫn trong README.md')

weights_ok = check_deca_weights(DECA_ROOT)
if weights_ok:
    print('✅ DECA weights: OK')
else:
    print('⚠️  DECA weights chưa được tải.')
    print('   Chạy: bash scripts/download_weights.sh')

## 3. Chạy DECA Inference

**Lưu ý**: Cell này chỉ hoạt động khi DECA đã được cài đặt đầy đủ với pretrained weights.
Nếu chưa có, xem phần 4 để load kết quả đã có sẵn.

In [ ]:
DEVICE = 'cuda'  # Đổi thành 'cpu' nếu không có GPU

if demo_script and weights_ok and INPUT_IMAGE.exists():
    print(f'🚀 Chạy DECA cho: {INPUT_IMAGE}')
    deca_output_dir = OUTPUT_DIR / 'deca'
    ensure_dir(deca_output_dir)

    return_code = run_deca(
        input_path=INPUT_IMAGE,
        output_dir=deca_output_dir,
        deca_root=DECA_ROOT,
        demo_script=demo_script,
        device=DEVICE,
        save_obj=True,
    )
    if return_code == 0:
        print('✅ DECA hoàn thành!')
    else:
        print(f'❌ DECA thất bại (code={return_code})')
else:
    print('⏭️  Bỏ qua inference (DECA chưa cài hoặc thiếu ảnh đầu vào).')
    print('   Xem kết quả mẫu ở phần tiếp theo nếu đã có outputs.')

## 4. Xem Kết quả DECA

In [ ]:
# Liệt kê các file output của DECA
deca_output_dir = OUTPUT_DIR / 'deca'
if deca_output_dir.exists():
    print('📁 Cấu trúc output DECA:')
    for p in sorted(deca_output_dir.rglob('*')):
        if p.is_file():
            size_kb = p.stat().st_size / 1024
            print(f'  {p.relative_to(deca_output_dir)} ({size_kb:.1f} KB)')
else:
    print(f'⚠️  Thư mục DECA output chưa tồn tại: {deca_output_dir}')
    print('   Chạy DECA inference ở cell trên trước.')

## 5. Hiển thị Rendered Image

In [ ]:
# Tìm và hiển thị ảnh rendered từ DECA
deca_output_dir = OUTPUT_DIR / 'deca'
rendered_images = list(deca_output_dir.rglob('*.png')) if deca_output_dir.exists() else []

if rendered_images:
    fig, axes = plt.subplots(1, min(3, len(rendered_images)), figsize=(15, 5))
    if len(rendered_images) == 1:
        axes = [axes]
    for ax, img_path in zip(axes, rendered_images[:3]):
        img = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img_rgb)
        ax.set_title(img_path.name, fontsize=10)
        ax.axis('off')
    plt.suptitle('DECA Output - DECA_DHMT', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('⚠️  Chưa có ảnh rendered. Hãy chạy DECA inference trước.')

## 6. Load và Visualize 3D Mesh

In [ ]:
# Load mesh .obj nếu có
obj_files = list(deca_output_dir.rglob('*.obj')) if deca_output_dir.exists() else []

if obj_files:
    obj_path = obj_files[0]
    print(f'Loading mesh: {obj_path}')
    vertices, faces = load_obj(obj_path)
    print(f'Vertices: {vertices.shape}')
    print(f'Faces: {faces.shape}')
    print(f'Vertex range: x=[{vertices[:,0].min():.3f}, {vertices[:,0].max():.3f}]')
    print(f'             y=[{vertices[:,1].min():.3f}, {vertices[:,1].max():.3f}]')
    print(f'             z=[{vertices[:,2].min():.3f}, {vertices[:,2].max():.3f}]')

    # Visualize với matplotlib
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    # Lấy mẫu một phần để hiển thị nhanh
    sample_idx = np.random.choice(len(vertices), min(5000, len(vertices)), replace=False)
    v = vertices[sample_idx]
    ax.scatter(v[:,0], v[:,1], v[:,2], c=v[:,1], cmap='plasma', s=0.5, alpha=0.6)
    ax.set_title('DECA 3D Face Mesh (sampled vertices)', fontsize=13)
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    plt.tight_layout()
    plt.show()
else:
    print('⚠️  Chưa có file .obj. Chạy DECA với --save-obj.')